# residuos_2 — productos "magicos" por WAPE en validacion (reemplaza la lista de z403)

Parte de `07_Residuo_sobre_baseline.ipynb` (a nivel producto-mes, mismos 6
baselines, mismos 6 esquemas, mismo control de leakage) y agrega lo que
`z403_RegresionLineal.ipynb` resolvia con una lista fija de ~180
`productos_magicos` sin documentar: **una regla reproducible, por producto**.

Para cada producto se compara, en los meses de VALIDACION, el WAPE del
esquema ganador contra el WAPE del baseline (el mismo `BASELINE` que ya elige
`07_Residuo_sobre_baseline` por validacion, entre `tn0`/`ma3`/`ma6`/`ma12`/
`ma_pond`/`lineal` -- no se inventa un "promedio" aparte). Si el esquema le
gana, el producto es magico y se le aplica el modelo complejo; si no, se le
aplica el baseline. El submit final mezcla las dos predicciones segun esta
tabla, calculada una sola vez y guardada para poder reusarla (`06_optuna_tres`
la lee para filtrar a pipe_nuevo).


In [ ]:
import gc, json, os, shutil, subprocess, time
from pathlib import Path

import numpy as np
import polars as pl
import pandas as pd
import lightgbm as lgb
import optuna
from sklearn.linear_model import Ridge

optuna.logging.set_verbosity(optuna.logging.WARNING)


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1", "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError("No encontre el bucket. Defini LABO3_BUCKET.")


def _leer_json_reintentando(path, intentos=5, espera=2):
    """El bucket es un mount GCS FUSE: a veces tira Input/output error transitorio."""
    ultimo_error = None
    for _ in range(intentos):
        try:
            with open(path, encoding="utf-8") as f:
                return json.load(f)
        except OSError as e:
            ultimo_error = e
            time.sleep(espera)
    raise ultimo_error


BUCKET   = resolver_bucket()
DIR_RAW  = BUCKET / "datasets"
DIR_FE   = BUCKET / "datasets_fe"     # aca se deja productos_magicos.* para 06_optuna_tres
RUTA_EXP = BUCKET / "exp_residuo"
RUTA_EXP.mkdir(parents=True, exist_ok=True)

print(f"BUCKET: {BUCKET}")
print(f"salida: {RUTA_EXP}")


def rango_meses(desde: int, hasta: int) -> list:
    a, b = (desde // 100) * 12 + desde % 100, (hasta // 100) * 12 + hasta % 100
    return [((m - 1) // 12) * 100 + ((m - 1) % 12) + 1 for m in range(a, b + 1)]


### Palancas (identicas a `07_Residuo_sobre_baseline.ipynb`, mas el umbral de "magico")


In [ ]:
PARAM = {
    # ── Datos ────────────────────────────────────────────────────────────
    'solo_productos_target': True,
    'muestra_productos': None,
    'horizonte': 2,
    'max_lags': 12,

    # ── Particion (misma que el pipe, para poder comparar) ───────────────
    'meses_train': rango_meses(201701, 201905),
    'meses_val':   [201907, 201908],
    'meses_test':  [201910],
    'reentrenar_con_val_para_test': True,

    # ── EL BASELINE ──────────────────────────────────────────────────────
    'baseline': 'auto',

    # ── EL ESQUEMA ───────────────────────────────────────────────────────
    'esquema': 'auto',

    # ── Optuna sobre el esquema ganador ──────────────────────────────────
    'n_trials': 40,
    'techo_arboles': 800,

    # ── Ridge ────────────────────────────────────────────────────────────
    'ridge_alpha': 1.0,

    # ── Productos magicos (NUEVO) ─────────────────────────────────────────
    # Un producto es 'magico' si wape_esquema < wape_baseline en VALIDACION,
    # medido solo con las filas de ESE producto. None = cualquier mejora
    # cuenta. Poner ej. 0.05 exige que la mejora relativa supere el 5% para
    # entrar (mas conservador: menos productos magicos, pero con mas margen).
    'mejora_minima_pct': None,

    # ── Fuente de la lista final (NUEVO) ───────────────────────────────────
    # 'calculado' -> la tabla de arriba (WAPE esquema vs baseline en validacion).
    # 'profesor'  -> ayuda de la catedra, guardada aparte (ver notebook, no
    #                se hardcodea aca). El calculo de todas formas se hace
    #                siempre, como diagnostico/comparacion -- solo cambia
    #                cual lista se USA para la mezcla/calibracion/submit.
    'fuente_magicos': 'profesor',
    'archivo_magicos_profesor': 'productos_magicos_profesor.json',

    # ── Entrega ──────────────────────────────────────────────────────────
    'periodo_objetivo': 202002,
    'semillas_ensemble': [102191],
    'clip_min': 0.0,
    'kaggle_competition': 'labo-iii-2026-rosario',
    'submit': False,
    'mensaje_submit': None,

    # ── Grilla completa: TODAS las combinaciones baseline x esquema (NUEVO) ──
    # True -> ademas de todo lo de arriba (que sigue eligiendo UN ganador por
    # validacion), corre las 6x6=36 combinaciones sin auto-elegir nada, y si
    # PARAM['submit']=True las sube TODAS a Kaggle -- sin importar el limite
    # diario de submits de la competencia. Las que no entren por el limite
    # quedan logueadas como fallidas, no cortan la corrida.
    'grilla_todas_las_combinaciones': False,
    # Pausa entre cada submit a Kaggle, para no golpear la API de una.
    'pausa_entre_submits_seg': 2,

    'semilla': 102191,
    'sufijo': '',
}

H = PARAM['horizonte']
L = PARAM['max_lags']

EXPERIMENTO = (f"residuo2_p_{L}lags_base-{PARAM['baseline']}_esq-{PARAM['esquema']}"
              f"_val{PARAM['meses_val'][0]}-{PARAM['meses_val'][-1]}"
              f"_test{PARAM['meses_test'][0]}"
              + (f"_{PARAM['sufijo']}" if PARAM['sufijo'] else ""))
DIR_OUT = RUTA_EXP / EXPERIMENTO
DIR_OUT.mkdir(parents=True, exist_ok=True)

print(f"EXPERIMENTO: {EXPERIMENTO}")
print(f"carpeta    : {DIR_OUT.relative_to(BUCKET)}")


### Panel producto-mes (identico a `07_Residuo_sobre_baseline.ipynb`)

Se colapsa la dimension cliente: es la misma agregacion que hace la metrica de la competencia antes de medir.


In [ ]:
t0 = time.time()

sell = pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t")
prod = (pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")
          .unique(subset=["product_id"]))
target_ids = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt")["product_id"].to_list()

if PARAM['solo_productos_target']:
    sell = sell.filter(pl.col("product_id").is_in(target_ids))
if PARAM['muestra_productos']:
    _top = (sell.group_by("product_id").agg(pl.col("tn").sum().alias("t"))
                .sort("t", descending=True).head(PARAM['muestra_productos'])["product_id"])
    sell = sell.filter(pl.col("product_id").is_in(_top.to_list()))

print(f"sell-in: {sell.height:,} filas · {sell['product_id'].n_unique()} productos "
      f"· {sell['customer_id'].n_unique()} clientes")

panel = (sell.group_by(["product_id", "periodo"])
             .agg(pl.col("tn").sum().alias("tn"),
                  pl.col("cust_request_tn").sum().alias("req_tn"),
                  pl.col("cust_request_qty").sum().alias("req_qty"),
                  pl.col("customer_id").n_unique().alias("n_clientes"),
                  pl.col("plan_precios_cuidados").max().alias("precios_cuidados"))
             .with_columns((((pl.col("periodo") // 100) * 12)
                            + (pl.col("periodo") % 100)).alias("m")))

vida = panel.group_by("product_id").agg(pl.col("m").min().alias("m_nace"),
                                        pl.col("m").max().alias("m_muere"))
grilla = (vida.with_columns(pl.int_ranges("m_nace", pl.col("m_muere") + 1).alias("m"))
              .explode("m").select("product_id", "m"))

panel = (grilla.join(panel.drop("periodo"), on=["product_id", "m"], how="left")
               .with_columns(pl.col("tn").fill_null(0.0), pl.col("req_tn").fill_null(0.0),
                             pl.col("req_qty").fill_null(0), pl.col("n_clientes").fill_null(0),
                             pl.col("precios_cuidados").fill_null(0))
               .join(vida, on="product_id", how="left")
               .join(prod.select("product_id", "cat1", "cat2", "cat3", "brand", "sku_size"),
                     on="product_id", how="left")
               .with_columns(
                   ((((pl.col("m") - 1) // 12) * 100) + ((pl.col("m") - 1) % 12) + 1)
                     .alias("periodo"),
                   pl.when(pl.col("m") >= pl.col("m_nace"))
                     .then(pl.col("m") - pl.col("m_nace")).otherwise(-1).alias("edad"))
               .sort(["product_id", "m"]))

print(f"panel: {panel.height:,} filas · {panel['product_id'].n_unique()} productos")
print(f"[{time.time()-t0:.0f}s]")


In [ ]:
for niv in ("cat1", "cat2", "cat3"):
    t = panel.group_by([niv, "m"]).agg(pl.col("tn").sum().alias(f"tn_{niv}"))
    panel = panel.join(t, on=[niv, "m"], how="left")
mercado = panel.group_by("m").agg(pl.col("tn").sum().alias("tn_mercado"))
panel = panel.join(mercado, on="m", how="left")


def div_segura(num, den, nombre):
    return (pl.when(pl.col(den).abs() > 1e-9)
              .then(pl.col(num) / pl.col(den)).otherwise(0.0).alias(nombre))


SHARES = [f"sh_{n}" for n in ("cat1", "cat2", "cat3", "mercado")]
panel = panel.with_columns([div_segura("tn", f"tn_{n}", f"sh_{n}")
                            for n in ("cat1", "cat2", "cat3", "mercado")])

df = panel.sort(["product_id", "m"]).with_columns(
    *[pl.col("tn").shift(k).over("product_id").alias(f"tn_lag{k}") for k in range(1, L + 1)],
    *[pl.col("tn").rolling_mean(w).over("product_id").alias(f"tn_ma{w}") for w in (3, 6, 12)],
    *[pl.col(s).shift(k).over("product_id").alias(f"{s}_lag{k}")
      for s in SHARES for k in (1, 2, 3)],
    *[pl.col(s).rolling_mean(3).over("product_id").alias(f"{s}_ma3") for s in SHARES],
    pl.col("n_clientes").shift(1).over("product_id").alias("n_clientes_lag1"),
    pl.col("n_clientes").rolling_mean(3).over("product_id").alias("n_clientes_ma3"),
    pl.col("req_qty").shift(1).over("product_id").alias("qty_lag1"),
    pl.col("tn").cum_max().over("product_id").alias("tn_pico_hasta_aca"),
    (pl.col("tn") > 0).cast(pl.Int8).alias("vendio"),
)

df = df.with_columns(
    *[(pl.col(s) - pl.col(f"{s}_lag1")).alias(f"{s}_d1") for s in SHARES],
    *[(pl.col(s) - pl.col(f"{s}_ma3")).alias(f"{s}_dma3") for s in SHARES],
    (pl.col("tn") - pl.col("tn_ma3")).alias("tn_dma3"),
    pl.col("vendio").rolling_mean(6).over("product_id").alias("frac_venta_6"),
    (pl.col("periodo") % 100).alias("mes_del_anio"),
    (pl.col("edad").is_between(0, 6)).cast(pl.Int8).alias("es_nuevo"),
)


def indice(num, den, nombre, techo=10.0):
    return (pl.when(pl.col(den).abs() > 1e-9)
              .then((pl.col(num) / pl.col(den)).clip(0.0, techo))
              .otherwise(pl.lit(None, dtype=pl.Float64)).alias(nombre))


df = df.with_columns(
    indice("tn", "tn_lag1", "idx_tn_mom"),
    indice("tn", "tn_ma3", "idx_tn_vs_ma3"),
    indice("tn", "tn_pico_hasta_aca", "idx_vs_pico"),
    indice("n_clientes", "n_clientes_lag1", "idx_clientes_mom"),
    indice("req_qty", "qty_lag1", "idx_qty_mom"),
)

df = df.sort(["product_id", "m"]).with_columns(
    pl.col("tn").shift(-H).over("product_id").alias("clase_tn"),
    ((((pl.col("m") + H - 1) // 12) * 100) + ((pl.col("m") + H - 1) % 12) + 1)
      .alias("periodo_objetivo"),
)

CATS = ["cat1", "cat2", "cat3", "brand"]
df = df.with_columns([pl.col(c).cast(pl.Utf8).fill_null("NA").cast(pl.Categorical)
                      for c in CATS])

NO_FEAT = {"product_id", "periodo", "m", "m_nace", "m_muere", "clase_tn",
          "periodo_objetivo"}
FEATURES = [c for c in df.columns if c not in NO_FEAT]

print(f"features: {len(FEATURES)}   filas: {df.height:,}")
print(f"con target: {int(df['clase_tn'].is_not_null().sum()):,}")


### Split y control de leakage (identico)


In [ ]:
sup = df.filter(pl.col("clase_tn").is_not_null())
periodos_sup = sorted(sup["periodo"].unique().to_list())
MESES_TRAIN = [m for m in PARAM['meses_train'] if m in periodos_sup]
MESES_VAL   = [m for m in PARAM['meses_val'] if m in periodos_sup]
MESES_TEST  = [m for m in PARAM['meses_test'] if m in periodos_sup]
MESES_INFER = sorted(df.filter(pl.col("clase_tn").is_null())["periodo"].unique().to_list())[-H:]
infer = df.filter(pl.col("periodo").is_in(MESES_INFER))

errores = []


def chk(ok, msg):
    print(f"  [{'ok   ' if ok else 'ERROR'}] {msg}")
    if not ok:
        errores.append(msg)


def a_m(p):
    return (p // 100) * 12 + (p % 100)


print("CONTROL DE LEAKAGE")
print("=" * 74)
for a, b, na, nb in ((MESES_TRAIN, MESES_VAL, "train", "val"),
                    (MESES_VAL, MESES_TEST, "val", "test")):
    g = a_m(min(b)) - a_m(max(a))
    chk(g >= H, f"gap {na}({max(a)}) -> {nb}({min(b)}) = {g} >= horizonte {H}")
if PARAM['reentrenar_con_val_para_test']:
    g = a_m(min(MESES_TEST)) - a_m(max(MESES_TRAIN + MESES_VAL))
    chk(g >= H, f"gap (train+val) -> test = {g} >= {H}")
chk(max(MESES_TRAIN) < min(MESES_VAL) < max(MESES_VAL) < min(MESES_TEST),
    "orden cronologico train < val < test")
chk("m_muere" not in FEATURES, "m_muere (dato del futuro) no es feature")
chk(not (set(FEATURES) & {"clase_tn", "periodo_objetivo"}), "el target no es feature")

_u = sup.group_by("product_id").agg(pl.len().alias("n")).sort("n", descending=True).head(1)
_s = df.filter(pl.col("product_id") == _u["product_id"][0]).sort("m")
_tn, _cl = _s["tn"].to_list(), _s["clase_tn"].to_list()
_mal = [i for i in range(len(_tn) - H)
       if _cl[i] is not None and abs(_cl[i] - _tn[i + H]) > 1e-9]
chk(not _mal, f"clase_tn[i] == tn[i+{H}] en el producto {_u['product_id'][0]} "
             f"({len(_tn)} meses, {len(_mal)} discrepancias)")

print("=" * 74)
if errores:
    raise RuntimeError(f"Leakage: {errores}")
print(f"TRAIN {len(MESES_TRAIN)} meses ({sup.filter(pl.col('periodo').is_in(MESES_TRAIN)).height:,} filas)"
     f" · VAL {MESES_VAL} · TEST {MESES_TEST} · INFER {MESES_INFER}")


### Los 6 baselines + los 6 esquemas (identico)


In [ ]:
def wape(y_real, y_pred, ids=None) -> float:
    yr = np.asarray(y_real, dtype=np.float64)
    yp = np.maximum(np.asarray(y_pred, dtype=np.float64), 0.0)
    if ids is not None:
        _, inv = np.unique(np.asarray(ids), return_inverse=True)
        yr, yp = np.bincount(inv, weights=yr), np.bincount(inv, weights=yp)
    den = np.abs(yr).sum()
    return float("nan") if den == 0 else float(np.abs(yr - yp).sum() / den)


def bloque(meses):
    return sup.filter(pl.col("periodo").is_in(meses))


tr, va, te = bloque(MESES_TRAIN), bloque(MESES_VAL), bloque(MESES_TEST)
COLS_LIN = ["tn"] + [f"tn_lag{k}" for k in range(1, L + 1)] + ["tn_ma3", "tn_ma6"]


def X_lin(b):
    return b.select(COLS_LIN).fill_null(0.0).to_numpy()


def wape_de(b, pred):
    return wape(b["clase_tn"].to_numpy(), pred, b["product_id"].to_numpy())


def baseline_fijo(b, cual):
    if cual == "tn0":
        return b["tn"].to_numpy().astype(np.float64)
    if cual == "ma_pond":
        return (0.5 * b["tn"].fill_null(0).to_numpy()
                + 0.3 * b["tn_lag1"].fill_null(0).to_numpy()
                + 0.2 * b["tn_lag2"].fill_null(0).to_numpy())
    return b[f"tn_{cual}"].fill_null(0.0).to_numpy().astype(np.float64)


_ridge_base = Ridge(alpha=PARAM['ridge_alpha'])
_ridge_base.fit(X_lin(tr), tr["clase_tn"].to_numpy())


def baseline_de(b, cual, ridge=None):
    if cual == "lineal":
        r = ridge if ridge is not None else _ridge_base
        return np.maximum(r.predict(X_lin(b)), 0.0)
    return np.maximum(baseline_fijo(b, cual), 0.0)


CANDIDATOS = ["tn0", "ma3", "ma6", "ma12", "ma_pond", "lineal"]
print(f"{'baseline':10s} {'WAPE val':>10s}")
print("-" * 22)
wape_base = {}
for c in CANDIDATOS:
    wape_base[c] = wape_de(va, baseline_de(va, c))
    print(f"{c:10s} {wape_base[c]:10.5f}")

BASELINE = (PARAM['baseline'] if PARAM['baseline'] != 'auto'
           else min(wape_base, key=wape_base.get))
print(f"\nbaseline elegido: {BASELINE}"
     + ("  (por validacion)" if PARAM['baseline'] == 'auto' else "  (forzado)"))


In [ ]:
PARAMS_LGBM = dict(objective="regression", metric="mae", verbosity=-1,
                   n_estimators=500, learning_rate=0.05, num_leaves=63,
                   min_child_samples=20, subsample=0.9, subsample_freq=1,
                   colsample_bytree=0.8, seed=PARAM['semilla'], n_jobs=-1,
                   deterministic=True, force_row_wise=True)


def fit_lgbm(b, target, params=None, lineal=False):
    p = dict(params or PARAMS_LGBM)
    if lineal:
        p.update(linear_tree=True, linear_lambda=1.0)
    m = lgb.LGBMRegressor(**p)
    bp = b.to_pandas()
    m.fit(bp[FEATURES], bp[target].to_numpy() if hasattr(bp[target], "to_numpy") else bp[target],
         categorical_feature=CATS)
    return m


def evaluar_esquemas(meses_fit, b_eval):
    fit = bloque(meses_fit)
    base_fit = baseline_de(fit, BASELINE)
    base_ev = baseline_de(b_eval, BASELINE)
    ev = b_eval.to_pandas()
    out, modelos = {}, {}

    out["A_baseline"] = base_ev

    mB = fit_lgbm(fit, "clase_tn")
    out["B_lgbm_nivel"] = mB.predict(ev[FEATURES]); modelos["B_lgbm_nivel"] = mB

    _num = [c for c in FEATURES if c not in CATS]
    mC = Ridge(alpha=PARAM['ridge_alpha'])
    mC.fit(fit.select(_num).fill_null(0.0).to_numpy(), fit["clase_tn"].to_numpy())
    out["C_lineal_nivel"] = mC.predict(b_eval.select(_num).fill_null(0.0).to_numpy())
    modelos["C_lineal_nivel"] = mC

    fit_d = fit.with_columns(
        pl.Series("y_res", fit["clase_tn"].to_numpy() - base_fit))
    mD = fit_lgbm(fit_d, "y_res")
    out["D_lgbm_residuo"] = base_ev + mD.predict(ev[FEATURES]); modelos["D_lgbm_residuo"] = mD

    rE = Ridge(alpha=PARAM['ridge_alpha'])
    rE.fit(X_lin(fit), fit["clase_tn"].to_numpy())
    base_lin_fit = np.maximum(rE.predict(X_lin(fit)), 0.0)
    base_lin_ev = np.maximum(rE.predict(X_lin(b_eval)), 0.0)
    fit_e = fit.with_columns(
        pl.Series("y_res", fit["clase_tn"].to_numpy() - base_lin_fit))
    mE = fit_lgbm(fit_e, "y_res")
    out["E_lineal_mas_lgbm"] = base_lin_ev + mE.predict(ev[FEATURES])
    modelos["E_lineal_mas_lgbm"] = (rE, mE)

    mF = fit_lgbm(fit, "clase_tn", lineal=True)
    out["F_lgbm_hojas_lineales"] = mF.predict(ev[FEATURES]); modelos["F_lgbm_hojas_lineales"] = mF

    return out, modelos


t0 = time.time()
pred_val, mod_val = evaluar_esquemas(MESES_TRAIN, va)
print(f"[{time.time()-t0:.0f}s]\n")

ESQUEMAS = list(pred_val)
print(f"{'esquema':24s} {'WAPE val':>10s}")
print("-" * 36)
wape_val = {}
for e in ESQUEMAS:
    wape_val[e] = wape_de(va, pred_val[e])
    print(f"{e:24s} {wape_val[e]:10.5f}")

ESQUEMA = (PARAM['esquema'] if PARAM['esquema'] != 'auto'
          else min(wape_val, key=wape_val.get))
print(f"\nesquema elegido: {ESQUEMA}"
     + ("  (por validacion)" if PARAM['esquema'] == 'auto' else "  (forzado)"))
_mej = 100 * (wape_val['A_baseline'] - wape_val[ESQUEMA]) / wape_val['A_baseline']
print(f"mejora sobre el baseline solo: {_mej:+.1f}%")


### Ridge del baseline lineal + Optuna sobre el esquema ganador (identico)


In [ ]:
_co = dict(zip(COLS_LIN, _ridge_base.coef_))
print(f"pesos de la Ridge (intercepto {_ridge_base.intercept_:+.3f}):")
for k, v in sorted(_co.items(), key=lambda kv: -abs(kv[1]))[:8]:
    print(f"   {k:10s} {v:+.4f}")


In [ ]:
def espacio(trial):
    return dict(
        objective="regression", metric="mae", verbosity=-1,
        seed=PARAM['semilla'], n_jobs=-1, subsample_freq=1,
        deterministic=True, force_row_wise=True,
        num_leaves=trial.suggest_int("num_leaves", 15, 255),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        learning_rate=trial.suggest_float("learning_rate", 5e-3, 0.2, log=True),
        n_estimators=trial.suggest_int("n_estimators", 200, PARAM['techo_arboles']),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 200),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    )


def predecir_esquema(esquema, meses_fit, b_eval, params, semilla=None):
    fit = bloque(meses_fit)
    ev = b_eval.to_pandas()
    p = dict(params)
    if semilla is not None:
        p["seed"] = semilla

    if esquema == "A_baseline":
        return baseline_de(b_eval, BASELINE), None

    if esquema == "C_lineal_nivel":
        _num = [c for c in FEATURES if c not in CATS]
        r = Ridge(alpha=PARAM['ridge_alpha'])
        r.fit(fit.select(_num).fill_null(0.0).to_numpy(), fit["clase_tn"].to_numpy())
        return r.predict(b_eval.select(_num).fill_null(0.0).to_numpy()), r

    if esquema == "B_lgbm_nivel":
        m = fit_lgbm(fit, "clase_tn", p)
        return m.predict(ev[FEATURES]), m

    if esquema == "F_lgbm_hojas_lineales":
        m = fit_lgbm(fit, "clase_tn", p, lineal=True)
        return m.predict(ev[FEATURES]), m

    if esquema == "D_lgbm_residuo":
        bf, be = baseline_de(fit, BASELINE), baseline_de(b_eval, BASELINE)
        f2 = fit.with_columns(pl.Series("y_res", fit["clase_tn"].to_numpy() - bf))
        m = fit_lgbm(f2, "y_res", p)
        return be + m.predict(ev[FEATURES]), m

    r = Ridge(alpha=PARAM['ridge_alpha'])
    r.fit(X_lin(fit), fit["clase_tn"].to_numpy())
    bf = np.maximum(r.predict(X_lin(fit)), 0.0)
    be = np.maximum(r.predict(X_lin(b_eval)), 0.0)
    f2 = fit.with_columns(pl.Series("y_res", fit["clase_tn"].to_numpy() - bf))
    m = fit_lgbm(f2, "y_res", p)
    return be + m.predict(ev[FEATURES]), (r, m)


SIN_HIPER = {"A_baseline", "C_lineal_nivel"}

if ESQUEMA in SIN_HIPER:
    print(f"El esquema ganador ({ESQUEMA}) no tiene hiperparametros que buscar.")
    study = None
    MEJORES = {}
else:
    study = optuna.create_study(
        direction="minimize", study_name=EXPERIMENTO,
        sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']),
        storage=f"sqlite:///{Path.home() / ('optuna_' + EXPERIMENTO + '.db')}",
        load_if_exists=True)

    def objective(trial):
        pred, _ = predecir_esquema(ESQUEMA, MESES_TRAIN, va, espacio(trial))
        return wape_de(va, pred)

    t0 = time.time()
    study.optimize(objective, n_trials=PARAM['n_trials'])
    MEJORES = {**espacio(optuna.trial.FixedTrial(study.best_params))}
    print(f"{len(study.trials)} trials · mejor WAPE val = {study.best_value:.5f}"
         f"  ({time.time()-t0:.0f}s)")
    for k, v in study.best_params.items():
        print(f"   {k:22s} {v}")


### Evaluacion en TEST (identico)


In [ ]:
MESES_FIT_TEST = (MESES_TRAIN + MESES_VAL) if PARAM['reentrenar_con_val_para_test'] else MESES_TRAIN
_pars = MEJORES or PARAMS_LGBM

pred_test, _ = evaluar_esquemas(MESES_FIT_TEST, te)
if MEJORES:
    pred_test[ESQUEMA], _ = predecir_esquema(ESQUEMA, MESES_FIT_TEST, te, _pars)

print(f"{'esquema':24s} {'WAPE val':>10s} {'WAPE test':>10s}")
print("-" * 48)
METRICAS = {}
for e in ESQUEMAS:
    wt = wape_de(te, pred_test[e])
    METRICAS[e] = {"val": wape_val[e], "test": wt}
    marca = "  <-" if e == ESQUEMA else ""
    print(f"{e:24s} {wape_val[e]:10.5f} {wt:10.5f}{marca}")

_a, _g = METRICAS['A_baseline']['test'], METRICAS[ESQUEMA]['test']
print(f"\n{ESQUEMA} vs baseline solo, en test: {100*(_a-_g)/_a:+.1f}%")


### Productos magicos (NUEVO): WAPE por producto en validacion

Reemplaza a la lista sin documentar de `z403_RegresionLineal.ipynb`. Por cada
producto, en los meses de VALIDACION: `wape_esquema` vs `wape_baseline`. Si
`wape_esquema < wape_baseline` (con el margen de `PARAM['mejora_minima_pct']`
si esta seteado), el producto es magico. Sin ventas en validacion -> no se
puede juzgar, queda NO magico (usa baseline) por seguridad.


In [ ]:
def wape_por_producto(df_ids_clase, pred, alias):
    """WAPE de CADA producto por separado (no agregado), vectorizado en polars."""
    return (df_ids_clase.with_columns(pl.Series("__pred", pred))
                       .group_by("product_id")
                       .agg((pl.col("clase_tn") - pl.col("__pred")).abs().sum().alias("abs_err"),
                            pl.col("clase_tn").abs().sum().alias("abs_real"),
                            pl.col("clase_tn").sum().alias("tn_val"))
                       .with_columns(
                           pl.when(pl.col("abs_real") > 1e-9)
                             .then(pl.col("abs_err") / pl.col("abs_real"))
                             .otherwise(None).alias(alias))
                       .select("product_id", alias, "tn_val"))


_ids_va = va.select("product_id", "clase_tn")
_wb = wape_por_producto(_ids_va, pred_val["A_baseline"], "wape_baseline")
_we = wape_por_producto(_ids_va, pred_val[ESQUEMA], "wape_esquema").select("product_id", "wape_esquema")

_umbral = PARAM['mejora_minima_pct']
productos_magicos = (
    _wb.join(_we, on="product_id", how="left")
       .with_columns(
           (100 * (pl.col("wape_baseline") - pl.col("wape_esquema")) / pl.col("wape_baseline"))
             .alias("mejora_pct"))
       .with_columns(
           ((pl.col("wape_esquema") < pl.col("wape_baseline"))
            & (pl.col("mejora_pct") >= (_umbral * 100 if _umbral else 0.0)))
             .fill_null(False).alias("magico"))
       .sort("tn_val", descending=True)
)

n_mag_calculado = int(productos_magicos["magico"].sum())
tn_mag_pct_calculado = 100 * (productos_magicos.filter(pl.col("magico"))["tn_val"].sum()
                              / productos_magicos["tn_val"].sum())
print(f"productos magicos (calculado): {n_mag_calculado} de {productos_magicos.height} "
     f"({100*n_mag_calculado/productos_magicos.height:.0f}%)  -  {tn_mag_pct_calculado:.0f}% del volumen de val")
print(productos_magicos.sort("mejora_pct", descending=True).head(10))

LISTA_CALCULADA = productos_magicos.filter(pl.col("magico"))["product_id"].to_list()

productos_magicos.write_parquet(DIR_FE / "productos_magicos.parquet")
with open(DIR_FE / "productos_magicos.json", "w", encoding="utf-8") as f:
    json.dump({
        "experimento": EXPERIMENTO, "baseline": BASELINE, "esquema": ESQUEMA,
        "mejora_minima_pct": _umbral,
        "n_magicos": n_mag_calculado, "n_total": productos_magicos.height,
        "tn_pct_magicos_val": round(tn_mag_pct_calculado, 1),
        "product_ids": [int(p) for p in LISTA_CALCULADA],
    }, f, indent=2, ensure_ascii=False)
print(f"Guardado (diagnostico, siempre se calcula): {DIR_FE / 'productos_magicos.parquet'}")

if PARAM['fuente_magicos'] == 'profesor':
    path_prof = DIR_FE / PARAM['archivo_magicos_profesor']
    if not path_prof.exists():
        raise FileNotFoundError(
            f"PARAM['fuente_magicos']='profesor' pero no encontre {path_prof}. "
            f"Guardala primero (JSON con clave 'product_ids'), o cambia a 'calculado'."
        )
    _meta_prof = _leer_json_reintentando(path_prof)
    LISTA_MAGICOS = [int(p) for p in _meta_prof['product_ids']]
    _solapa = set(LISTA_CALCULADA) & set(LISTA_MAGICOS)
    print(f"\nusando lista del PROFESOR ({path_prof.name}): {len(LISTA_MAGICOS)} productos")
    print(f"solapamiento profesor vs calculada: {len(_solapa)} en comun "
         f"(calculada: {len(LISTA_CALCULADA)}, profesor: {len(LISTA_MAGICOS)})")
else:
    LISTA_MAGICOS = LISTA_CALCULADA
    print(f"\nusando lista CALCULADA: {len(LISTA_MAGICOS)} productos")

n_mag_usado = len(LISTA_MAGICOS)
tn_mag_pct_usado = (100 * productos_magicos.filter(pl.col("product_id").is_in(LISTA_MAGICOS))["tn_val"].sum()
                    / productos_magicos["tn_val"].sum()) if productos_magicos.height else 0.0
print(f"lista USADA para mezcla/calibracion/submit: {n_mag_usado} productos "
     f"({tn_mag_pct_usado:.0f}% del volumen de val)")
print(f"\nGuardado: {DIR_FE / 'productos_magicos.json'}  (diagnostico calculado)")


### Prediccion mezclada (NUEVO): magico -> esquema, resto -> baseline

Se mide en TEST (la lista de magicos se definio con VALIDACION, nunca con
test) para ver si mezclar ayuda de verdad o si conviene quedarse con el
esquema puro.


In [ ]:
_ids_te = te.select("product_id", "clase_tn").with_columns(
    pl.Series("pred_baseline", pred_test["A_baseline"]),
    pl.Series("pred_esquema", pred_test[ESQUEMA]),
    pl.col("product_id").is_in(LISTA_MAGICOS).alias("magico"),
).with_columns(
    pl.when(pl.col("magico")).then(pl.col("pred_esquema"))
      .otherwise(pl.col("pred_baseline")).alias("pred_mezcla")
)

wape_mezcla_test = wape(_ids_te["clase_tn"], _ids_te["pred_mezcla"], _ids_te["product_id"])
print(f"{'variante':24s} {'WAPE test':>10s}")
print("-" * 36)
print(f"{'A_baseline (todo)':24s} {METRICAS['A_baseline']['test']:10.5f}")
print(f"{ESQUEMA + ' (todo)':24s} {METRICAS[ESQUEMA]['test']:10.5f}")
print(f"{'mezcla (magicos+resto)':24s} {wape_mezcla_test:10.5f}")

_mej_mezcla = 100 * (METRICAS[ESQUEMA]['test'] - wape_mezcla_test) / METRICAS[ESQUEMA]['test']
print(f"\nmezcla vs {ESQUEMA} puro: {_mej_mezcla:+.2f}%"
     + ("  <- la mezcla ayuda" if _mej_mezcla > 0 else "  <- el esquema puro ya era mejor"))


### Calibracion de sesgo (NUEVO)

El mejor resultado real hasta ahora (Stepwise + factor 1.02) sugiere que
una correccion GLOBAL de sesgo, aplicada DESPUES de predecir, puede aportar
mas que agregar complejidad al modelo. Se mide un factor unico
`real_total / predicho_total` en VALIDACION sobre la mezcla, y se prueba en
TEST antes de aplicarlo -- si no mejora el WAPE test, se deja el factor en
1.0 (no se aplica a ciegas solo porque 'suena bien').


In [ ]:
pred_val_mezcla = (va.select("product_id").with_columns(
    pl.Series("pred_baseline", pred_val["A_baseline"]),
    pl.Series("pred_esquema", pred_val[ESQUEMA]),
    pl.col("product_id").is_in(LISTA_MAGICOS).alias("magico"),
).with_columns(
    pl.when(pl.col("magico")).then(pl.col("pred_esquema"))
      .otherwise(pl.col("pred_baseline")).alias("pred_mezcla")
)["pred_mezcla"])

_real_val_total = float(va["clase_tn"].sum())
_pred_val_total = float(pred_val_mezcla.sum())
FACTOR_CALIBRACION = _real_val_total / _pred_val_total if _pred_val_total > 1e-9 else 1.0
print(f"real total (val)     : {_real_val_total:,.1f}")
print(f"predicho total (val) : {_pred_val_total:,.1f}")
print(f"factor de calibracion: {FACTOR_CALIBRACION:.4f}")

wape_mezcla_test_calibrado = wape(_ids_te["clase_tn"], _ids_te["pred_mezcla"] * FACTOR_CALIBRACION,
                                  _ids_te["product_id"])
print(f"\n{'variante':24s} {'WAPE test':>10s}")
print("-" * 36)
print(f"{'mezcla sin calibrar':24s} {wape_mezcla_test:10.5f}")
print(f"{'mezcla calibrada':24s} {wape_mezcla_test_calibrado:10.5f}")

USAR_CALIBRACION = wape_mezcla_test_calibrado < wape_mezcla_test
FACTOR_FINAL = FACTOR_CALIBRACION if USAR_CALIBRACION else 1.0
print(f"\n{'SE APLICA' if USAR_CALIBRACION else 'NO se aplica'} la calibracion al submit final "
     f"(factor={FACTOR_FINAL:.4f})"
     + ("" if USAR_CALIBRACION else
        f"  -- empeoraba el WAPE test ({wape_mezcla_test_calibrado:.5f} > {wape_mezcla_test:.5f})"))


### Reentreno con todos los meses supervisados, para la entrega


In [ ]:
MESES_TODOS = sorted(periodos_sup)
print(f"reentrenando {ESQUEMA} con {len(MESES_TODOS)} meses "
     f"({MESES_TODOS[0]}..{MESES_TODOS[-1]})")

preds = []
for sem in PARAM['semillas_ensemble']:
    p, _ = predecir_esquema(ESQUEMA, MESES_TODOS, infer, _pars, semilla=sem)
    preds.append(p)
    print(f"  semilla {sem} lista")
pred_infer_esquema = np.maximum(np.mean(preds, axis=0), PARAM['clip_min'])
pred_infer_baseline = baseline_de(infer, BASELINE)

pred_infer = infer.select("product_id", "periodo", "periodo_objetivo").with_columns(
    pl.Series("tn_pred_esquema", pred_infer_esquema),
    pl.Series("tn_pred_baseline", pred_infer_baseline),
    pl.col("product_id").is_in(LISTA_MAGICOS).alias("magico"),
).with_columns(
    pl.when(pl.col("magico")).then(pl.col("tn_pred_esquema"))
      .otherwise(pl.col("tn_pred_baseline")).alias("tn_pred")
)
pred_infer.write_parquet(DIR_OUT / "predicciones_inferencia.parquet")

print(f"\npredicciones: {pred_infer.height:,} filas "
     f"({int(pred_infer['magico'].sum())} filas de productos magicos)")
print(pred_infer.group_by("periodo", "periodo_objetivo").len().sort("periodo"))
print(f"\ntn_pred   min {pred_infer['tn_pred'].min():.2f}   media {pred_infer['tn_pred'].mean():.2f}   "
     f"max {pred_infer['tn_pred'].max():.2f}")


### Entrega (con la mezcla) contra la lista oficial


In [ ]:
OBJ = PARAM['periodo_objetivo']
obj = pred_infer.filter(pl.col("periodo_objetivo") == OBJ)
if obj.is_empty():
    raise RuntimeError(f"No hay predicciones para {OBJ}. Disponibles: "
                       f"{sorted(pred_infer['periodo_objetivo'].unique().to_list())}")

por_producto = obj.group_by("product_id").agg(pl.col("tn_pred").sum().alias("tn"))
oficiales = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt", separator="\t")
submit = oficiales.select("product_id").join(por_producto, on="product_id", how="left")
sin_pred = int(submit["tn"].null_count())
submit = submit.with_columns(pl.col("tn").fill_null(0.0)).sort("product_id")
submit = submit.with_columns((pl.col("tn") * FACTOR_FINAL).alias("tn"))
print(f"factor de calibracion aplicado al submit: {FACTOR_FINAL:.4f}")

print(f"mes objetivo {OBJ}: {obj.height} filas -> {por_producto.height} productos")
print(f"lista oficial: {oficiales.height}   sin prediccion (van en 0): {sin_pred}")
if sin_pred > oficiales.height * 0.05:
    print("   ATENCION: mas del 5% de la lista. Revisalo antes de subir.")
print(f"\ntn   min {submit['tn'].min():.3f}   media {submit['tn'].mean():.3f}   "
     f"max {submit['tn'].max():.3f}   suma {submit['tn'].sum():,.1f}")
print(submit.head(10))

path_submit = DIR_OUT / f"submission_{OBJ}_mezcla.csv"
submit.write_csv(path_submit)
shutil.copy(path_submit, RUTA_EXP / "submission_ultima_mezcla.csv")
print(f"\nGuardado: {path_submit}")


### Submit a Kaggle (opcional)


In [ ]:
def kaggle_cli(args):
    try:
        r = subprocess.run(["kaggle"] + args, capture_output=True, text=True)
        return r.returncode == 0, (r.stdout or "") + (r.stderr or "")
    except FileNotFoundError:
        return False, "La CLI de kaggle no esta instalada.  pip install kaggle"
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"


if not PARAM['submit']:
    print("PARAM['submit'] = False -> no se sube. El CSV ya esta generado.")
else:
    kd = Path.home() / ".kaggle" / "kaggle.json"
    kd.parent.mkdir(parents=True, exist_ok=True)
    if not kd.exists():
        for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
            if cand.exists():
                shutil.copy(cand, kd); kd.chmod(0o600); break
    if not kd.exists():
        print("\nSin credenciales de Kaggle. El CSV ya esta generado.")
    else:
        kd.chmod(0o600)
        msg = PARAM['mensaje_submit'] or (
            f"{ESQUEMA}+mezcla sobre {BASELINE} | wape_test={wape_mezcla_test:.5f} "
            f"| {n_mag_usado} magicos ({PARAM['fuente_magicos']}) | factor_calibracion={FACTOR_FINAL:.4f}")
        ok, salida = kaggle_cli(["competitions", "submit",
                                "-c", PARAM['kaggle_competition'],
                                "-f", str(path_submit), "-m", msg])
        print(f"\nmensaje: {msg}\n{salida}")
        print("Submit enviado." if ok else "NO se pudo subir; el CSV esta en disco.")


### `resultado.json` + leaderboard


In [ ]:
resultado = {
    'experimento': EXPERIMENTO,
    'idea': 'residuo sobre baseline + productos magicos por WAPE en validacion (reemplaza '
            'la lista sin documentar de z403)',
    'granularidad': 'producto-mes',
    'baseline_elegido': BASELINE, 'wape_baselines_val': wape_base,
    'esquema_elegido': ESQUEMA, 'metricas_por_esquema': METRICAS,
    'fuente_magicos': PARAM['fuente_magicos'],
    'n_productos_magicos': n_mag_usado, 'n_productos_total': productos_magicos.height,
    'tn_pct_magicos_val': round(tn_mag_pct_usado, 1),
    'n_productos_magicos_calculado': n_mag_calculado,
    'n_solapa_profesor_calculado': len(set(LISTA_CALCULADA) & set(LISTA_MAGICOS)),
    'wape_test_esquema_puro': METRICAS[ESQUEMA]['test'],
    'wape_test_baseline_puro': METRICAS['A_baseline']['test'],
    'wape_test_mezcla': wape_mezcla_test,
    'mejora_mezcla_vs_esquema_pct': round(_mej_mezcla, 2),
    'factor_calibracion': FACTOR_FINAL,
    'wape_test_mezcla_calibrado': wape_mezcla_test_calibrado,
    'calibracion_aplicada': USAR_CALIBRACION,
    'horizonte': H, 'max_lags': L,
    'meses_train': MESES_TRAIN, 'meses_val': MESES_VAL, 'meses_test': MESES_TEST,
    'meses_inferencia': MESES_INFER, 'periodo_objetivo': OBJ,
    'n_features': len(FEATURES), 'features': FEATURES,
    'n_trials': len(study.trials) if study else 0,
    'hiperparametros': study.best_params if study else {},
    'pesos_ridge_baseline': {k: round(float(v), 5) for k, v in _co.items()},
    'ridge_alpha': PARAM['ridge_alpha'],
    'n_sin_prediccion': sin_pred,
    'tn_total': float(submit['tn'].sum()),
    'archivo_productos_magicos': str(DIR_FE / (PARAM['archivo_magicos_profesor']
                                              if PARAM['fuente_magicos'] == 'profesor'
                                              else "productos_magicos.json")),
    'semilla': PARAM['semilla'],
}
with open(DIR_OUT / "resultado.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False, default=str)

fila = {'experimento': EXPERIMENTO, 'baseline': BASELINE, 'esquema': ESQUEMA,
       'max_lags': L, 'n_features': len(FEATURES),
       'n_magicos': n_mag_usado, 'n_total': productos_magicos.height,
       'fuente_magicos': PARAM['fuente_magicos'],
       'wape_test_mezcla': round(wape_mezcla_test, 5),
       'wape_test_esquema_puro': round(METRICAS[ESQUEMA]['test'], 5),
       'wape_test_baseline_puro': round(METRICAS['A_baseline']['test'], 5),
       'mejora_mezcla_vs_esquema_pct': round(_mej_mezcla, 2),
       'factor_calibracion': round(FACTOR_FINAL, 4),
       'wape_test_mezcla_calibrado': round(wape_mezcla_test_calibrado, 5),
       'sin_prediccion': sin_pred, 'tn_total': round(float(submit['tn'].sum()), 1)}

path_lb = RUTA_EXP / "leaderboard_residuo.csv"
nueva = pl.DataFrame([fila])
if path_lb.exists():
    viejo = pl.read_csv(path_lb).filter(pl.col("experimento") != EXPERIMENTO)
    nueva = pl.concat([viejo, nueva], how="diagonal_relaxed")
nueva.sort("wape_test_mezcla").write_csv(path_lb)

print(f"Archivos en {DIR_OUT.relative_to(BUCKET)}:")
for p in sorted(DIR_OUT.iterdir()):
    print(f"  - {p.name}")
print(f"\nleaderboard_residuo.csv ({nueva.height} experimentos):")
print(nueva.select("baseline", "esquema", "n_magicos", "wape_test_mezcla",
                   "wape_test_esquema_puro", "mejora_mezcla_vs_esquema_pct"))


### Grilla completa: TODAS las combinaciones baseline x esquema (NUEVO, opcional)

No reemplaza nada de arriba (que sigue eligiendo UN ganador por validacion,
con Optuna y la mezcla con productos magicos) -- esto es aparte, gateado por
`PARAM['grilla_todas_las_combinaciones']`. Corre las 6x6=36 combinaciones SIN
auto-elegir: mide WAPE val y test de cada una, arma un submit POR
combinacion, y si `PARAM['submit']=True` las sube TODAS a Kaggle, sin
importar el limite diario de submits de la competencia -- las que no entren
quedan logueadas como fallidas, no cortan la corrida.

Reusa `evaluar_esquemas()` UNA VEZ POR BASELINE (6 llamadas, no 36): esa
funcion ya devuelve las 6 predicciones de esquema para el baseline activo. Los
esquemas con LightGBM usan siempre `PARAMS_LGBM` (los defaults, NO los
hiperparametros que Optuna afino para el combo ganador de arriba -- serian
36 busquedas de Optuna, demasiado caro): esta grilla compara arquitecturas
con hiperparametros por default, no el resultado final ya afinado.


In [ ]:
if PARAM['grilla_todas_las_combinaciones']:
    t0_grilla = time.time()
    N_COMBOS = len(CANDIDATOS) * len(ESQUEMAS)
    print(f"Grilla completa: {len(CANDIDATOS)} baselines x {len(ESQUEMAS)} esquemas "
         f"= {N_COMBOS} combinaciones")

    filas_grilla = []
    pred_infer_por_combo = {}

    for b_actual in CANDIDATOS:
        BASELINE = b_actual   # evaluar_esquemas/baseline_de/predecir_esquema leen esta global
        pv, _ = evaluar_esquemas(MESES_TRAIN, va)
        pt, _ = evaluar_esquemas(MESES_FIT_TEST, te)
        for e_actual in ESQUEMAS:
            wv, wt = wape_de(va, pv[e_actual]), wape_de(te, pt[e_actual])
            filas_grilla.append({'baseline': b_actual, 'esquema': e_actual,
                                 'wape_val': round(wv, 5), 'wape_test': round(wt, 5)})
            p_inf, _ = predecir_esquema(e_actual, MESES_TODOS, infer, PARAMS_LGBM)
            pred_infer_por_combo[(b_actual, e_actual)] = np.maximum(
                np.asarray(p_inf, dtype=np.float64), PARAM['clip_min'])
        print(f"  baseline={b_actual}: listo  [{time.time()-t0_grilla:.0f}s acumulado]")

    grilla = pl.DataFrame(filas_grilla).sort("wape_test")
    grilla.write_csv(DIR_OUT / "grilla_combinaciones.csv")
    print(f"\nGuardado: {DIR_OUT / 'grilla_combinaciones.csv'}")
    print(grilla)
else:
    print("PARAM['grilla_todas_las_combinaciones'] = False -> no se corre la grilla.")


### Los 36 CSV de entrega (siempre se generan, suba o no suba a Kaggle)


In [ ]:
if PARAM['grilla_todas_las_combinaciones']:
    paths_grilla = {}
    for (b_actual, e_actual), pred_arr in pred_infer_por_combo.items():
        tag = f"{b_actual}_{e_actual}"
        tmp = infer.select("product_id", "periodo_objetivo").with_columns(
            pl.Series("tn_pred", pred_arr))
        obj_g = tmp.filter(pl.col("periodo_objetivo") == OBJ)
        por_prod_g = obj_g.group_by("product_id").agg(pl.col("tn_pred").sum().alias("tn"))
        submit_g = (oficiales.select("product_id")
                             .join(por_prod_g, on="product_id", how="left")
                             .with_columns(pl.col("tn").fill_null(0.0)).sort("product_id"))
        path_g = DIR_OUT / f"submission_{OBJ}_{tag}.csv"
        submit_g.write_csv(path_g)
        paths_grilla[(b_actual, e_actual)] = path_g

    print(f"{len(paths_grilla)} CSV de entrega generados en {DIR_OUT.relative_to(BUCKET)}/")


### Submit de las 36 a Kaggle (opcional, sin importar el limite diario)


In [ ]:
if PARAM['grilla_todas_las_combinaciones']:
    _submits_grilla = []
    if not PARAM['submit']:
        print("PARAM['submit'] = False -> no se sube nada. Los CSV ya estan generados.")
    else:
        kd = Path.home() / ".kaggle" / "kaggle.json"
        kd.parent.mkdir(parents=True, exist_ok=True)
        if not kd.exists():
            for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
                if cand.exists():
                    shutil.copy(cand, kd); kd.chmod(0o600); break
        if not kd.exists():
            print("Sin credenciales de Kaggle. Los CSV ya estan generados, no se sube nada.")
        else:
            kd.chmod(0o600)
            print(f"Subiendo {len(paths_grilla)} submits a Kaggle "
                 f"(competencia: {PARAM['kaggle_competition']})...")
            for i, ((b_actual, e_actual), path_g) in enumerate(paths_grilla.items(), 1):
                tag = f"{b_actual}_{e_actual}"
                _fila_g = grilla.filter((pl.col("baseline") == b_actual)
                                        & (pl.col("esquema") == e_actual)).to_dicts()[0]
                msg = f"grilla {tag} | wape_val={_fila_g['wape_val']} wape_test={_fila_g['wape_test']}"
                ok, salida = kaggle_cli(["competitions", "submit",
                                        "-c", PARAM['kaggle_competition'],
                                        "-f", str(path_g), "-m", msg])
                _submits_grilla.append({'combo': tag, 'ok': ok, 'mensaje': msg,
                                        'salida': salida.strip()[:300]})
                print(f"  [{i}/{len(paths_grilla)}] {tag}: "
                     f"{'OK' if ok else 'FALLO (revisar salida abajo)'}")
                if not ok:
                    print(f"      {salida.strip()[:200]}")
                time.sleep(PARAM['pausa_entre_submits_seg'])

            n_ok = sum(1 for s in _submits_grilla if s['ok'])
            print(f"\n{n_ok} de {len(_submits_grilla)} submits OK.")
            with open(DIR_OUT / "grilla_submits.json", "w", encoding="utf-8") as f:
                json.dump(_submits_grilla, f, indent=2, ensure_ascii=False)
            print(f"Registro: {DIR_OUT / 'grilla_submits.json'}")
